# 06. Fusion Pipeline

Build a 12-feature late-fusion dataset from the trained EEG, MEG, speech, and face models, then train a logistic-regression meta-classifier.


In [1]:
from pathlib import Path
import sys

def _find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "NeuroSense" / "webdev" / "backend").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root for this notebook.")

PROJECT_ROOT = _find_project_root()
NOTEBOOKS_DIR = PROJECT_ROOT / "NeuroSense" / "notebooks"
BACKEND_DIR = PROJECT_ROOT / "NeuroSense" / "webdev" / "backend"

for path in (NOTEBOOKS_DIR, BACKEND_DIR):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from notebook_support import bootstrap_notebook

ctx = bootstrap_notebook(PROJECT_ROOT)
DATASETS_DIR = ctx["datasets_dir"]
ARTIFACTS_DIR = ctx["artifacts_dir"]
CACHE_DIR = ctx["cache_dir"]
RANDOM_STATE = ctx["random_state"]

print(f"Project root: {PROJECT_ROOT}")
print(f"Datasets directory: {DATASETS_DIR}")
print(f"Artifacts directory: {ARTIFACTS_DIR}")


Project root: /Users/devashishsingh/Desktop/human emotion recognition system
Datasets directory: /Users/devashishsingh/Desktop/human emotion recognition system/NeuroSense/datasets
Artifacts directory: /Users/devashishsingh/Desktop/human emotion recognition system/NeuroSense/artifacts


In [2]:
import joblib
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from utils.emotion_utils import SENTIMENT_ORDER, aggregate_probabilities, map_emotion_to_sentiment

REQUIRED_FILES = [
    ARTIFACTS_DIR / "eeg" / "eeg_model.pkl",
    ARTIFACTS_DIR / "eeg" / "eeg_scaler.pkl",
    ARTIFACTS_DIR / "eeg" / "eeg_label_encoder.pkl",
    ARTIFACTS_DIR / "meg" / "meg_model.pkl",
    ARTIFACTS_DIR / "meg" / "meg_scaler.pkl",
    ARTIFACTS_DIR / "meg" / "meg_label_encoder.pkl",
    ARTIFACTS_DIR / "speech" / "speech_model.pkl",
    ARTIFACTS_DIR / "speech" / "speech_scaler.pkl",
    ARTIFACTS_DIR / "speech" / "speech_label_encoder.pkl",
    ARTIFACTS_DIR / "face" / "face_model.pkl",
    ARTIFACTS_DIR / "face" / "face_scaler.pkl",
    ARTIFACTS_DIR / "face" / "face_pca.pkl",
    ARTIFACTS_DIR / "face" / "face_label_encoder.pkl",
    CACHE_DIR / "speech_features.npz",
    CACHE_DIR / "face_test_sentiment.npz",
]

missing = [str(path) for path in REQUIRED_FILES if not path.exists()]
if missing:
    raise FileNotFoundError("Run notebooks 01, 02, 04, and 05 before notebook 06. Missing files:\n" + "\n".join(missing))


In [3]:
eeg_model = joblib.load(ARTIFACTS_DIR / "eeg" / "eeg_model.pkl")
eeg_scaler = joblib.load(ARTIFACTS_DIR / "eeg" / "eeg_scaler.pkl")
eeg_encoder = joblib.load(ARTIFACTS_DIR / "eeg" / "eeg_label_encoder.pkl")

meg_model = joblib.load(ARTIFACTS_DIR / "meg" / "meg_model.pkl")
meg_scaler = joblib.load(ARTIFACTS_DIR / "meg" / "meg_scaler.pkl")
meg_encoder = joblib.load(ARTIFACTS_DIR / "meg" / "meg_label_encoder.pkl")

speech_model = joblib.load(ARTIFACTS_DIR / "speech" / "speech_model.pkl")
speech_scaler = joblib.load(ARTIFACTS_DIR / "speech" / "speech_scaler.pkl")
speech_encoder = joblib.load(ARTIFACTS_DIR / "speech" / "speech_label_encoder.pkl")

face_model = joblib.load(ARTIFACTS_DIR / "face" / "face_model.pkl")
face_scaler = joblib.load(ARTIFACTS_DIR / "face" / "face_scaler.pkl")
face_pca = joblib.load(ARTIFACTS_DIR / "face" / "face_pca.pkl")
face_encoder = joblib.load(ARTIFACTS_DIR / "face" / "face_label_encoder.pkl")

eeg_df = pd.read_csv(DATASETS_DIR / "eeg" / "eeg" / "emotions.csv")
eeg_X = eeg_df.drop(columns=["label"]).select_dtypes(include=[np.number]).fillna(0.0).values
eeg_y = eeg_df["label"].astype(str).values
_, eeg_X_test, _, eeg_y_test = train_test_split(
    eeg_X,
    eeg_y,
    test_size=0.2,
    stratify=eeg_y,
    random_state=RANDOM_STATE,
)
eeg_probs = eeg_model.predict_proba(eeg_scaler.transform(eeg_X_test))

meg_df = pd.read_csv(DATASETS_DIR / "meg" / "meg_features.csv")
meg_X = meg_df.drop(columns=["label"]).values
meg_y = meg_df["label"].astype(str).values
_, meg_X_test, _, meg_y_test = train_test_split(
    meg_X,
    meg_y,
    test_size=0.2,
    stratify=meg_y,
    random_state=RANDOM_STATE,
)
meg_probs = meg_model.predict_proba(meg_scaler.transform(meg_X_test))

speech_cache = np.load(CACHE_DIR / "speech_features.npz", allow_pickle=True)
speech_X = speech_cache["X"]
speech_y = speech_cache["y"].astype(str)
_, speech_X_test, _, speech_y_test = train_test_split(
    speech_X,
    speech_y,
    test_size=0.2,
    stratify=speech_y,
    random_state=RANDOM_STATE,
)
speech_probs = speech_model.predict_proba(speech_scaler.transform(speech_X_test))

face_cache = np.load(CACHE_DIR / "face_test_sentiment.npz", allow_pickle=True)
face_X_test = face_cache["X"]
face_y_test = face_cache["y"].astype(str)
face_probs = face_model.predict_proba(face_pca.transform(face_scaler.transform(face_X_test)))

print("EEG test samples:", len(eeg_y_test))
print("MEG test samples:", len(meg_y_test))
print("Speech test samples:", len(speech_y_test))
print("Face test samples:", len(face_y_test))


/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:153: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:153: RuntimeWarning: overflow encountered in matmul
  X_transformed = X @ self.components_.T
/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:153: RuntimeWarning: invalid value encountered in matmul
  X_transformed = X @ self.components_.T


EEG test samples: 427
MEG test samples: 120
Speech test samples: 210
Face test samples: 1500


In [4]:
def build_probability_pool(true_labels, probability_matrix, class_labels):
    pool = {sentiment: [] for sentiment in SENTIMENT_ORDER}
    for label, row in zip(true_labels, probability_matrix):
        sentiment = map_emotion_to_sentiment(label)
        pooled_row = aggregate_probabilities(class_labels, row, order=SENTIMENT_ORDER)
        pool[sentiment].append(pooled_row)

    return {
        sentiment: np.vstack(rows)
        for sentiment, rows in pool.items()
        if rows
    }

modality_pools = {
    "eeg": build_probability_pool(eeg_y_test, eeg_probs, eeg_encoder.classes_),
    "meg": build_probability_pool(meg_y_test, meg_probs, meg_encoder.classes_),
    "speech": build_probability_pool(speech_y_test, speech_probs, speech_encoder.classes_),
    "face": build_probability_pool(face_y_test, face_probs, face_encoder.classes_),
}

for modality, pool in modality_pools.items():
    counts = {sentiment: rows.shape[0] for sentiment, rows in pool.items()}
    print(modality, counts)

for modality, pool in modality_pools.items():
    for sentiment in SENTIMENT_ORDER:
        if sentiment not in pool:
            raise ValueError(f"Fusion pool for {modality} is missing sentiment {sentiment}")

rng = np.random.default_rng(RANDOM_STATE)
SAMPLES_PER_SENTIMENT = 250
meta_features = []
meta_labels = []

for sentiment in SENTIMENT_ORDER:
    for _ in range(SAMPLES_PER_SENTIMENT):
        parts = []
        for modality in ("eeg", "meg", "speech", "face"):
            rows = modality_pools[modality][sentiment]
            parts.append(rows[rng.integers(len(rows))])
        meta_features.append(np.concatenate(parts))
        meta_labels.append(sentiment)

X_meta = np.vstack(meta_features).astype(np.float32)
y_meta_raw = np.asarray(meta_labels)

fusion_encoder = LabelEncoder()
y_meta = fusion_encoder.fit_transform(y_meta_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X_meta,
    y_meta,
    test_size=0.2,
    stratify=y_meta,
    random_state=RANDOM_STATE,
)

fusion_model = LogisticRegression(max_iter=2000, multi_class="auto", random_state=RANDOM_STATE)
fusion_model.fit(X_train, y_train)
y_pred = fusion_model.predict(X_test)

print("Fusion feature matrix:", X_meta.shape)
print(f"Fusion test accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=fusion_encoder.classes_))


eeg {'NEGATIVE': 142, 'NEUTRAL': 143, 'POSITIVE': 142}
meg {'NEGATIVE': 42, 'NEUTRAL': 39, 'POSITIVE': 39}
speech {'NEGATIVE': 120, 'NEUTRAL': 30, 'POSITIVE': 60}
face {'NEGATIVE': 500, 'NEUTRAL': 500, 'POSITIVE': 500}
Fusion feature matrix: (750, 12)
Fusion test accuracy: 1.0000
              precision    recall  f1-score   support

    NEGATIVE       1.00      1.00      1.00        50
     NEUTRAL       1.00      1.00      1.00        50
    POSITIVE       1.00      1.00      1.00        50

    accuracy                           1.00       150
   macro avg       1.00      1.00      1.00       150
weighted avg       1.00      1.00      1.00       150



/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:167: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:167: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:167: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:300: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n

In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="crest",
    xticklabels=fusion_encoder.classes_,
    yticklabels=fusion_encoder.classes_,
)
plt.title("Fusion confusion matrix")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.show()

artifact_dir = ARTIFACTS_DIR / "fusion"
artifact_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(fusion_model, artifact_dir / "fusion_model.pkl")
joblib.dump(fusion_encoder, artifact_dir / "fusion_label_encoder.pkl")

print("Saved fusion artifacts to:", artifact_dir)


Saved fusion artifacts to: /Users/devashishsingh/Desktop/human emotion recognition system/NeuroSense/artifacts/fusion


/var/folders/y1/kwl357md29bd8ggvss23h_yc0000gn/T/ipykernel_59814/3768788483.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
